# 🍜 Fine-tune `ThuyNT03/PhoBERT_NER_T4-ABSA` cho TikTok Vietnamese F&B ABSA (`M5.2`)

Notebook độc lập chạy trên **Kaggle GPU (T4 x2 / P100)** để fine-tune toàn bộ 12 tầng Transformer của [`ThuyNT03/PhoBERT_NER_T4-ABSA`](https://huggingface.co/ThuyNT03/PhoBERT_NER_T4-ABSA) kết hợp nhúng thanh điệu tiếng Việt `NFC` cho bài toán **4-Aspect Sentiment + Exact NFC Character Spans** (`taste`, `price`, `hygiene`, `service`).

### Cách chạy trên Kaggle (3 bước — ~2 phút):
1. Bật **Accelerator: GPU T4 x2 (hoặc P100)** và bật **Internet: ON** ở thanh Settings bên phải.
2. Bấm **Add Input / Upload Dataset** → tải file `tiktok_fnb_absa_dataset.zip` lên (đặt tên dataset bất kỳ, Notebook sẽ tự quét `/kaggle/input` để tìm file).
3. Bấm **Run All** → Khi chạy xong Cell cuối, tải file **`/kaggle/working/phobert_ner_t4_absa_finetuned_bundle.zip`** về máy và gửi lại vào project!


In [ ]:
import hashlib
import json
import math
import os
import random
import re
import time
import unicodedata
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import save_file as save_safetensors
from transformers import AutoModelForTokenClassification, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def find_input_file(filename: str) -> Path:
    search_roots = [Path("/kaggle/input"), Path("./dataset"), Path("."), Path("../data/annotations")]
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Không tìm thấy {filename}! Hãy Add Input file tiktok_fnb_absa_dataset.zip vào Kaggle.")


TRAIN_PATH = find_input_file("absa_train_video_disjoint_v1.jsonl")
POOL_PATH = find_input_file("m5_2_pilot_pool.jsonl")
print("Found TRAIN_PATH:", TRAIN_PATH)
print("Found POOL_PATH :", POOL_PATH)


def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(ln) for ln in path.read_text(encoding="utf-8").splitlines() if ln.strip()]


def split_units_by_video(rows: list[dict], dev_ratio: float = 0.5, seed: int = 42) -> dict[str, list[dict]]:
    unique_vids = sorted({str(r["video_id"]).strip() for r in rows})
    rng = np.random.default_rng(seed)
    shuffled = list(unique_vids)
    rng.shuffle(shuffled)
    n_dev = max(1, min(len(unique_vids) - 1, round(len(unique_vids) * dev_ratio)))
    dev_vids = set(shuffled[:n_dev])
    return {
        "dev": [r for r in rows if str(r["video_id"]).strip() in dev_vids],
        "held_out": [r for r in rows if str(r["video_id"]).strip() not in dev_vids],
    }


disjoint_train_rows = load_jsonl(TRAIN_PATH)
pool_rows = load_jsonl(POOL_PATH)
splits = split_units_by_video(pool_rows, seed=42)
dev_rows = splits["dev"]
held_out_rows = splits["held_out"]

train_vids = {str(r["video_id"]) for r in disjoint_train_rows}
dev_vids = sorted({str(r["video_id"]) for r in dev_rows})
held_vids = {str(r["video_id"]) for r in held_out_rows}

assert len(train_vids & held_vids) == 0, "LEAKAGE: train overlaps held_out!"
assert len(set(dev_vids) & held_vids) == 0, "LEAKAGE: dev overlaps held_out!"
print(
    f"Zero-Leakage Verified!\n"
    f"  Disjoint Train : {len(disjoint_train_rows)} comments across {len(train_vids)} videos\n"
    f"  Human Dev      : {len(dev_rows)} units across {len(dev_vids)} videos\n"
    f"  Locked Held-Out: {len(held_out_rows)} units across {len(held_vids)} unseen videos (0 overlap)"
)


In [ ]:
BASE_MODEL_ID = "ThuyNT03/PhoBERT_NER_T4-ABSA"
ASPECT_NAMES = ("taste", "price", "hygiene", "service")
ASPECT_STATE_LABELS = ("absent_unknown", "positive", "neutral", "negative")
ASPECT_STATE_TO_ID = {lbl: idx for idx, lbl in enumerate(ASPECT_STATE_LABELS)}
OBSERVED_POLARITIES = ("positive", "neutral", "negative")

BIO_LABELS = ("O",) + tuple(
    f"{prefix}-{asp}-{pol}"
    for asp in ASPECT_NAMES
    for pol in OBSERVED_POLARITIES
    for prefix in ("B", "I")
)
BIO_LABEL_TO_ID = {lbl: idx for idx, lbl in enumerate(BIO_LABELS)}
WORD_BOUNDARY_RE = re.compile(r"\S+")


def _hash_ngram_bucket(ngram: str, num_buckets: int) -> int:
    digest = hashlib.blake2b(ngram.encode("utf-8"), digest_size=4).digest()
    val = int.from_bytes(digest, "little")
    return 1 + (val % max(1, num_buckets - 1))


def _expand_to_full_word(text: str, s: int, e: int) -> str:
    n = len(text)
    ws = max(0, min(s, n))
    we = max(ws, min(e, n))
    while ws > 0 and not text[ws - 1].isspace():
        ws -= 1
    while we < n and not text[we].isspace():
        we += 1
    return text[ws:we]


def encode_token_diacritic_hashes(
    text: str,
    offsets: list[tuple[int, int]],
    *,
    num_buckets: int = 8192,
    max_ngrams: int = 12,
) -> torch.Tensor:
    nfc = unicodedata.normalize("NFC", text or "").lower()
    out = torch.zeros((len(offsets), max_ngrams), dtype=torch.long)
    for idx, (s, e) in enumerate(offsets):
        if e <= s or s < 0 or e > len(nfc):
            continue
        tok_slice = nfc[s:e].strip()
        full_word = _expand_to_full_word(nfc, s, e).strip(".,!?;:\"'()[]{}…")
        if not tok_slice and not full_word:
            continue
        feats = []
        if tok_slice:
            feats.append(f"t:{tok_slice}")
        if full_word:
            padded = f"<{full_word}>"
            feats.append(f"w:{padded}")
            for k in (2, 3, 4):
                for j in range(max(0, len(padded) - k + 1)):
                    feats.append(padded[j : j + k])
                    if len(feats) >= max_ngrams:
                        break
                if len(feats) >= max_ngrams:
                    break
        for j, fg in enumerate(feats[:max_ngrams]):
            out[idx, j] = _hash_ngram_bucket(fg, num_buckets)
    return out


def align_char_spans_to_bio_ids(offsets: list[tuple[int, int]], spans: list[dict]) -> list[int]:
    labels = [-100 if e_tok <= s_tok else 0 for s_tok, e_tok in offsets]
    for sp in spans:
        aspect = str(sp.get("aspect") or "").strip()
        pol = str(sp.get("polarity_label") or "").strip()
        if aspect not in ASPECT_NAMES or pol not in OBSERVED_POLARITIES:
            continue
        sp_s, sp_e = int(sp.get("start_char", 0)), int(sp.get("end_char", 0))
        if sp_e <= sp_s:
            continue
        b_id = BIO_LABEL_TO_ID[f"B-{aspect}-{pol}"]
        i_id = BIO_LABEL_TO_ID[f"I-{aspect}-{pol}"]
        started = False
        for idx, (s_tok, e_tok) in enumerate(offsets):
            if e_tok <= s_tok:
                continue
            if min(e_tok, sp_e) - max(s_tok, sp_s) > 0:
                labels[idx] = b_id if not started else i_id
                started = True
    return labels


class PhoBertNerT4MultiTaskABSA(nn.Module):
    def __init__(
        self,
        *,
        backbone: nn.Module,
        pretrained_ner_head: nn.Linear,
        diacritic_buckets: int = 8192,
        diacritic_dim: int = 192,
        fusion_dim: int = 384,
        dropout_prob: float = 0.15,
    ) -> None:
        super().__init__()
        self.backbone = backbone
        self.backbone.pooler = None
        self.pretrained_ner_head = pretrained_ner_head
        self.diacritic_buckets = diacritic_buckets
        self.diacritic_dim = diacritic_dim
        self.fusion_dim = fusion_dim

        self.diacritic_emb = nn.Embedding(diacritic_buckets, diacritic_dim, padding_idx=0)
        hidden_size = int(backbone.config.hidden_size)
        ner_dim = int(pretrained_ner_head.out_features)

        self.fusion_proj = nn.Sequential(
            nn.Linear(hidden_size + ner_dim + diacritic_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout_prob),
        )
        self.bio_classifier = nn.Linear(fusion_dim, len(BIO_LABELS))
        self.aspect_queries = nn.Parameter(torch.randn(len(ASPECT_NAMES), fusion_dim) * 0.02)
        self.aspect_classifiers = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(fusion_dim * 2, fusion_dim // 2),
                    nn.GELU(),
                    nn.Dropout(dropout_prob),
                    nn.Linear(fusion_dim // 2, len(ASPECT_STATE_LABELS)),
                )
                for _ in ASPECT_NAMES
            ]
        )

    @classmethod
    def from_pretrained_ner_t4(cls, model_id: str = BASE_MODEL_ID) -> "PhoBertNerT4MultiTaskABSA":
        base_tc = AutoModelForTokenClassification.from_pretrained(model_id)
        backbone = base_tc.bert
        backbone.pooler = None
        ner_head = nn.Linear(base_tc.classifier.in_features, base_tc.classifier.out_features)
        with torch.no_grad():
            ner_head.weight.copy_(base_tc.classifier.weight)
            ner_head.bias.copy_(base_tc.classifier.bias)
        return cls(backbone=backbone, pretrained_ner_head=ner_head)

    def forward(
        self,
        *,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        diacritic_ngram_ids: torch.Tensor,
        bio_labels: torch.Tensor | None = None,
        aspect_labels: torch.Tensor | None = None,
        aspect_class_weights: torch.Tensor | None = None,
        bio_class_weights: torch.Tensor | None = None,
    ) -> dict:
        bert_out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden = bert_out.last_hidden_state
        ner_logits = self.pretrained_ner_head(hidden)

        emb = self.diacritic_emb(diacritic_ngram_ids)
        non_pad = (diacritic_ngram_ids != 0).unsqueeze(-1).to(emb.dtype)
        diac_vec = (emb * non_pad).sum(dim=2) / non_pad.sum(dim=2).clamp(min=1.0)

        fused = self.fusion_proj(torch.cat([hidden, ner_logits, diac_vec], dim=-1))
        bio_logits = self.bio_classifier(fused)

        tok_mask = attention_mask.bool()
        scale = float(self.fusion_dim) ** -0.5
        attn_scores = torch.einsum("af,blf->bal", self.aspect_queries, fused) * scale
        attn_scores = attn_scores.masked_fill(~tok_mask.unsqueeze(1), -1e4)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_pooled = torch.einsum("bal,blf->baf", attn_weights, fused)

        max_pooled = fused.masked_fill(~tok_mask.unsqueeze(-1), -1e4).max(dim=1).values
        aspect_logits = torch.stack(
            [head(torch.cat([attn_pooled[:, i, :], max_pooled], dim=-1)) for i, head in enumerate(self.aspect_classifiers)],
            dim=1,
        )

        res = {
            "bio_logits": bio_logits,
            "aspect_logits": aspect_logits,
            "attn_weights": attn_weights,
        }
        if bio_labels is not None and aspect_labels is not None:
            bio_loss = F.cross_entropy(
                bio_logits.view(-1, bio_logits.size(-1)),
                bio_labels.view(-1),
                weight=bio_class_weights,
                ignore_index=-100,
            )
            asp_loss = sum(
                F.cross_entropy(
                    aspect_logits[:, i, :],
                    aspect_labels[:, i],
                    weight=(aspect_class_weights[i] if aspect_class_weights is not None else None),
                )
                for i in range(len(ASPECT_NAMES))
            ) / float(len(ASPECT_NAMES))
            res["loss"] = 0.45 * bio_loss + 0.55 * asp_loss
        return res


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
model = PhoBertNerT4MultiTaskABSA.from_pretrained_ner_t4(BASE_MODEL_ID).to(DEVICE)

# Split 14 dev videos into 7 dev_train (upweighted human labels) + 7 dev_val (unseen checkpoint/threshold selection)
half_dev = len(dev_vids) // 2
dev_train_vids = set(dev_vids[:half_dev])
dev_val_vids = set(dev_vids[half_dev:])
dev_train_rows = [dict(r, _is_human_dev=True) for r in dev_rows if str(r["video_id"]) in dev_train_vids]
dev_val_rows = [r for r in dev_rows if str(r["video_id"]) in dev_val_vids]


def encode_rows(rows: list[dict], aspect_col: str, spans_col: str, max_length: int = 96) -> list[dict]:
    items = []
    for r in rows:
        nfc = unicodedata.normalize("NFC", str(r.get("text") or ""))
        if not nfc.strip():
            continue
        enc = tokenizer(nfc, return_offsets_mapping=True, truncation=True, max_length=max_length)
        offsets = [(int(a), int(b)) for a, b in enc["offset_mapping"]]
        diac = encode_token_diacritic_hashes(nfc, offsets, num_buckets=8192, max_ngrams=12)
        bio_ids = align_char_spans_to_bio_ids(offsets, r.get(spans_col) or [])
        asp_map = r.get(aspect_col) or {}
        asp_ids = [ASPECT_STATE_TO_ID[str(asp_map.get(a, "absent_unknown"))] for a in ASPECT_NAMES]
        has_rare = any(str(asp_map.get(a, "absent_unknown")) != "absent_unknown" for a in ("price", "hygiene", "service")) or (
            str(asp_map.get("taste", "absent_unknown")) in {"neutral", "negative"}
        )
        items.append(
            {
                "input_ids": [int(x) for x in enc["input_ids"]],
                "diacritic_ids": diac,
                "bio_ids": bio_ids,
                "aspect_ids": asp_ids,
                "has_any": any(x != 0 for x in asp_ids),
                "has_rare": has_rare,
                "is_human_dev": bool(r.get("_is_human_dev", False)),
            }
        )
    return items


enc_train = encode_rows(disjoint_train_rows, "aspects", "spans") + encode_rows(
    dev_train_rows, "adjudicated_aspects", "adjudicated_spans"
)
pos_items = [x for x in enc_train if x["has_any"]]
rare_items = [x for x in enc_train if x["has_rare"]]
human_items = [x for x in enc_train if x["is_human_dev"]]
neg_items = [x for x in enc_train if not x["has_any"]]


def collate_gpu(batch_items: list[dict], pad_id: int) -> dict[str, torch.Tensor]:
    max_len = max(len(x["input_ids"]) for x in batch_items)
    bsz = len(batch_items)
    inp = torch.full((bsz, max_len), pad_id, dtype=torch.long, device=DEVICE)
    msk = torch.zeros((bsz, max_len), dtype=torch.long, device=DEVICE)
    diac = torch.zeros((bsz, max_len, 12), dtype=torch.long, device=DEVICE)
    bio = torch.full((bsz, max_len), -100, dtype=torch.long, device=DEVICE)
    asp = torch.zeros((bsz, 4), dtype=torch.long, device=DEVICE)
    for i, it in enumerate(batch_items):
        L = len(it["input_ids"])
        inp[i, :L] = torch.tensor(it["input_ids"], dtype=torch.long, device=DEVICE)
        msk[i, :L] = 1
        diac[i, :L, :] = it["diacritic_ids"][:L].to(DEVICE)
        bio[i, :L] = torch.tensor(it["bio_ids"], dtype=torch.long, device=DEVICE)
        asp[i, :] = torch.tensor(it["aspect_ids"], dtype=torch.long, device=DEVICE)
    return {"input_ids": inp, "attention_mask": msk, "diacritic_ngram_ids": diac, "bio_labels": bio, "aspect_labels": asp}


aspect_w = []
for idx in range(4):
    cnt = Counter(x["aspect_ids"][idx] for x in enc_train)
    aspect_w.append(
        torch.tensor(
            [
                1.0,
                min(4.5, math.sqrt(max(1, cnt[0]) / max(1, cnt.get(1, 1))) * 0.45),
                min(5.5, math.sqrt(max(1, cnt[0]) / max(1, cnt.get(2, 1))) * 0.45),
                min(5.0, math.sqrt(max(1, cnt[0]) / max(1, cnt.get(3, 1))) * 0.45),
            ],
            dtype=torch.float32,
            device=DEVICE,
        )
    )
aspect_class_weights = torch.stack(aspect_w, dim=0)
bio_w = torch.ones(len(BIO_LABELS), dtype=torch.float32, device=DEVICE)
bio_w[0] = 0.25

# Unfreeze ALL 12 Transformer layers on GPU with differential learning rate!
bert_params = [p for n, p in model.named_parameters() if n.startswith("backbone.")]
head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
optimizer = torch.optim.AdamW(
    [
        {"params": bert_params, "lr": 2.5e-5, "weight_decay": 0.01},
        {"params": head_params, "lr": 8.0e-4, "weight_decay": 0.01},
    ]
)


def eval_dev_f1(curr_model: nn.Module, eval_rows: list[dict], thresh: float = 0.50) -> float:
    curr_model.eval()
    f1s = []
    preds_all = []
    with torch.inference_mode():
        for r in eval_rows:
            nfc = unicodedata.normalize("NFC", str(r["text"]))
            enc = tokenizer(nfc, return_offsets_mapping=True, truncation=True, max_length=96, return_tensors="pt")
            offsets = [(int(a), int(b)) for a, b in enc["offset_mapping"][0].tolist()]
            diac = encode_token_diacritic_hashes(nfc, offsets).unsqueeze(0).to(DEVICE)
            out = curr_model(
                input_ids=enc["input_ids"].to(DEVICE),
                attention_mask=enc["attention_mask"].to(DEVICE),
                diacritic_ngram_ids=diac,
            )
            probs = torch.softmax(out["aspect_logits"][0], dim=-1).cpu().tolist()
            preds_all.append(probs)
    for i, asp in enumerate(ASPECT_NAMES):
        gold = [str(r["adjudicated_aspects"][asp]) != "absent_unknown" for r in eval_rows]
        if not any(gold):
            continue
        tp = sum(1 for p, g in zip(preds_all, gold) if (1.0 - p[i][0] >= thresh) and g)
        fp = sum(1 for p, g in zip(preds_all, gold) if (1.0 - p[i][0] >= thresh) and not g)
        fn = sum(1 for p, g in zip(preds_all, gold) if (1.0 - p[i][0] < thresh) and g)
        pr = tp / (tp + fp) if (tp + fp) else 0.0
        rc = tp / (tp + fn) if (tp + fn) else 0.0
        f1s.append((2 * pr * rc / (pr + rc)) if (pr + rc) else 0.0)
    return sum(f1s) / max(1, len(f1s))


EPOCHS = 15
BATCH_SIZE = 32
rng_py = random.Random(SEED)
pad_id = int(tokenizer.pad_token_id or 0)
total_steps = EPOCHS * math.ceil((len(pos_items) + len(rare_items) * 2 + len(human_items) * 3 + 600) / BATCH_SIZE)
step = 0
best_dev_f1 = -1.0
best_state = None
epoch_logs = []
t_train_start = time.perf_counter()

for ep in range(1, EPOCHS + 1):
    t_ep = time.perf_counter()
    model.train()
    pool = (
        list(pos_items)
        + list(rare_items) * 2
        + list(human_items) * 3
        + rng_py.sample(neg_items, k=min(600, len(neg_items)))
    )
    rng_py.shuffle(pool)
    loss_sum, nb = 0.0, 0
    for s in range(0, len(pool), BATCH_SIZE):
        batch = collate_gpu(pool[s : s + BATCH_SIZE], pad_id)
        step += 1
        warm = max(1, int(0.10 * total_steps))
        mult = (step / warm) if step <= warm else (0.12 + 0.88 * 0.5 * (1.0 + math.cos(math.pi * (step - warm) / max(1, total_steps - warm))))
        optimizer.param_groups[0]["lr"] = 2.5e-5 * mult
        optimizer.param_groups[1]["lr"] = 8.0e-4 * mult

        optimizer.zero_grad()
        out = model(**batch, aspect_class_weights=aspect_class_weights, bio_class_weights=bio_w)
        out["loss"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        loss_sum += float(out["loss"].item())
        nb += 1

    dev_f1 = eval_dev_f1(model, dev_val_rows)
    ep_sec = round(time.perf_counter() - t_ep, 2)
    mean_l = round(loss_sum / max(1, nb), 4)
    if dev_f1 >= best_dev_f1:
        best_dev_f1 = dev_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    epoch_logs.append({"epoch": ep, "mean_loss": mean_l, "dev_val_aspect_f1": round(dev_f1, 4), "wall_seconds": ep_sec})
    print(f"[Epoch {ep:02d}/{EPOCHS}] loss={mean_l:.4f} | dev_val_F1={dev_f1:.4f} (best={best_dev_f1:.4f}) | {ep_sec}s")

train_wall_sec = round(time.perf_counter() - t_train_start, 2)
if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
model.eval()
print(f"Training finished in {train_wall_sec}s! Restored best dev_val checkpoint (F1={best_dev_f1:.4f}).")


In [ ]:
def predict_raw_on_device(text: str) -> dict:
    nfc = unicodedata.normalize("NFC", text or "")
    enc = tokenizer(nfc, return_offsets_mapping=True, truncation=True, max_length=96, return_tensors="pt")
    offsets = [(int(a), int(b)) for a, b in enc["offset_mapping"][0].tolist()]
    diac = encode_token_diacritic_hashes(nfc, offsets).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        out = model(
            input_ids=enc["input_ids"].to(DEVICE),
            attention_mask=enc["attention_mask"].to(DEVICE),
            diacritic_ngram_ids=diac,
        )
        asp_probs = torch.softmax(out["aspect_logits"][0], dim=-1).cpu().tolist()
        bio_ids = out["bio_logits"][0].argmax(dim=-1).cpu().tolist()
        attn_w = out["attn_weights"][0].cpu().tolist()
    return {"nfc": nfc, "offsets": offsets, "asp_probs": asp_probs, "bio_ids": bio_ids, "attn_w": attn_w}


# Calibrate per-aspect thresholds ONLY on dev_rows (0 held_out exposure!)
dev_raw = [predict_raw_on_device(r["text"]) for r in dev_rows]
thresholds = {}
for idx, asp in enumerate(ASPECT_NAMES):
    gold_obs = [str(r["adjudicated_aspects"][asp]) != "absent_unknown" for r in dev_rows]
    if not any(gold_obs):
        thresholds[asp] = 0.65
        continue
    best_f1, best_t = -1.0, 0.50
    for cand_t in (0.30, 0.36, 0.42, 0.48, 0.54, 0.60, 0.66, 0.72, 0.78):
        tp = sum(1 for rw, g in zip(dev_raw, gold_obs) if (1.0 - rw["asp_probs"][idx][0] >= cand_t) and g)
        fp = sum(1 for rw, g in zip(dev_raw, gold_obs) if (1.0 - rw["asp_probs"][idx][0] >= cand_t) and not g)
        fn = sum(1 for rw, g in zip(dev_raw, gold_obs) if (1.0 - rw["asp_probs"][idx][0] < cand_t) and g)
        pr = tp / (tp + fp) if (tp + fp) else 0.0
        rc = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * pr * rc / (pr + rc)) if (pr + rc) else 0.0
        if f1 > best_f1 or (math.isclose(f1, best_f1) and cand_t > best_t):
            best_f1, best_t = f1, cand_t
    thresholds[asp] = best_t

print("Calibrated dev thresholds:", thresholds)

# Evaluate pure neural checkpoint on locked unseen held_out (93 human-verified units)
held_raw = [predict_raw_on_device(r["text"]) for r in held_out_rows]
det_f1s = []
sent_correct, sent_total = 0, 0
for idx, asp in enumerate(ASPECT_NAMES):
    t_asp = thresholds[asp]
    gold_st = [str(r["adjudicated_aspects"][asp]) for r in held_out_rows]
    pred_obs = [(1.0 - rw["asp_probs"][idx][0]) >= t_asp for rw in held_raw]
    tp = sum(1 for p, g in zip(pred_obs, gold_st) if p and g != "absent_unknown")
    fp = sum(1 for p, g in zip(pred_obs, gold_st) if p and g == "absent_unknown")
    fn = sum(1 for p, g in zip(pred_obs, gold_st) if (not p) and g != "absent_unknown")
    if (tp + fn) > 0:
        pr = tp / (tp + fp) if (tp + fp) else 0.0
        rc = tp / (tp + fn) if (tp + fn) else 0.0
        det_f1s.append((2 * pr * rc / (pr + rc)) if (pr + rc) else 0.0)
    for rw, g in zip(held_raw, gold_st):
        if g != "absent_unknown":
            sent_total += 1
            best_pol_i = max((1, 2, 3), key=lambda k: rw["asp_probs"][idx][k])
            if ASPECT_STATE_LABELS[best_pol_i] == g:
                sent_correct += 1

pure_det_macro_f1 = round(sum(det_f1s) / max(1, len(det_f1s)), 4)
pure_sent_acc = round(sent_correct / max(1, sent_total), 4)
print(f"Unseen Held-Out (Pure Neural): aspect_detection_macro_F1={pure_det_macro_f1}, gold_observed_polarity_acc={pure_sent_acc}")

# Save pickle-free safetensors bundle + zip archive in /kaggle/working/
work_root = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("./kaggle_output")
bundle_out = work_root / "phobert-ner-t4-absa-v1"
bundle_out.mkdir(parents=True, exist_ok=True)

tokenizer.save_pretrained(str(bundle_out))
model.backbone.config.save_pretrained(str(bundle_out))
cpu_state = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()}
weights_file = bundle_out / "model.safetensors"
save_safetensors(cpu_state, str(weights_file))
weights_sha256 = hashlib.sha256(weights_file.read_bytes()).hexdigest()

meta = {
    "base_model_id": BASE_MODEL_ID,
    "architecture": "PhoBertNerT4MultiTaskABSA",
    "ner_num_labels": int(model.pretrained_ner_head.out_features),
    "diacritic_buckets": model.diacritic_buckets,
    "diacritic_dim": model.diacritic_dim,
    "fusion_dim": model.fusion_dim,
    "max_length": 96,
    "max_ngrams": 12,
    "thresholds": thresholds,
    "model_safetensors_sha256": weights_sha256,
    "model_safetensors_bytes": weights_file.stat().st_size,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "trained_on_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(DEVICE),
    "epochs": EPOCHS,
    "unfreeze_top_k_layers": 12,
    "train_wall_seconds": train_wall_sec,
}
(bundle_out / "absa_model_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

kaggle_report = {
    "meta": meta,
    "epoch_logs": epoch_logs,
    "pure_neural_held_out_preview": {
        "aspect_detection_macro_f1_supported": pure_det_macro_f1,
        "gold_observed_polarity_accuracy": pure_sent_acc,
    },
}
report_path = work_root / "training_metrics_kaggle.json"
report_path.write_text(json.dumps(kaggle_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

zip_out = work_root / "phobert_ner_t4_absa_finetuned_bundle.zip"
with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(bundle_out.iterdir()):
        if fpath.is_file():
            zf.write(fpath, arcname=f"phobert-ner-t4-absa-v1/{fpath.name}")
    zf.write(report_path, arcname="training_metrics_kaggle.json")

print("=" * 72)
print("HOÀN TẤT! HÃY TẢI FILE NÀY VỀ MÁY VÀ GỬI LẠI VÀO PROJECT:")
print(f"  -> {zip_out} ({zip_out.stat().st_size / (1024 * 1024):.2f} MB)")
print("=" * 72)
